# SimCLR + ResNet50-ViT + Attention-MIL on PI-CAI mpMRI
**CSC 8851 Deep Learning — Spring 2026**  
Hemanth Kumar Mulluri · Shiv Brahmbhatt · Georgia State University  
**Real PI-CAI data — Fold 0 (295 patients)**


## Cell 1 — Install dependencies

In [ ]:
!pip install -q monai==1.3.0 SimpleITK scikit-learn

## Cell 2 — Mount Drive + Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, random, json, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
import pandas as pd
import SimpleITK as sitk

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU — switch runtime to A100')

SAVE_DIR = Path('/content/drive/MyDrive/csPCa_checkpoints')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Checkpoints: {SAVE_DIR}')

## Cell 3 — Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────
IMG_DIR   = Path('/content/drive/MyDrive/picai_public_images_fold0')
LABEL_CSV = Path('/content/drive/MyDrive/Saved from the Google app/Metadata with lesion info.csv')

# ── Image settings ───────────────────────────────────────────────────────
IMG_SIZE   = 224
N_CHANNELS = 3   # t2w, adc, hbv

# ── Encoder ──────────────────────────────────────────────────────────────
EMBED_DIM  = 768
VIT_HEADS  = 8
VIT_LAYERS = 6
PROJ_DIM   = 128
ATTN_DIM   = 256

# ── SimCLR ───────────────────────────────────────────────────────────────
SSL_EPOCHS      = 30
SSL_BATCH       = 64
SSL_LR          = 3e-4
SSL_TEMP        = 0.07
SSL_ES_PATIENCE = 5

# ── MIL ──────────────────────────────────────────────────────────────────
MIL_EPOCHS   = 30
MIL_LR       = 1e-4
MIL_WD       = 0.01
MIL_PATIENCE = 5

VAL_SPLIT = 0.2   # 80/20 train/val split from fold0

## Cell 4 — Dataset

In [ ]:
class PiCAIFold0Dataset(Dataset):
    """
    Loads PI-CAI patients from fold0 .mha files.
    Channels: T2W, ADC, HBV (high b-value DWI).
    Label: case_csPCa (1 = csPCa ISUP>=2, 0 = benign).
    """
    def __init__(self, patient_ids, img_dir, label_map):
        self.ids      = patient_ids
        self.img_dir  = Path(img_dir)
        self.lmap     = label_map
        self.resize   = transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True)

    def _load_mha(self, path):
        img = sitk.ReadImage(str(path))
        arr = sitk.GetArrayFromImage(img).astype(np.float32)  # (D, H, W)
        t   = torch.from_numpy(arr)
        # per-volume z-score normalisation
        t = (t - t.mean()) / (t.std() + 1e-6)
        return t

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        pid = str(self.ids[idx])
        d   = self.img_dir / pid

        # find the study files
        files = os.listdir(d)
        t2w_f = next(f for f in files if f.endswith('_t2w.mha'))
        adc_f = next(f for f in files if f.endswith('_adc.mha'))
        hbv_f = next(f for f in files if f.endswith('_hbv.mha'))

        t2w = self._load_mha(d / t2w_f)
        adc = self._load_mha(d / adc_f)
        hbv = self._load_mha(d / hbv_f)

        # align depths
        D = min(t2w.shape[0], adc.shape[0], hbv.shape[0])
        t2w, adc, hbv = t2w[:D], adc[:D], hbv[:D]

        # stack as (D, 3, H, W)
        bag = torch.stack([t2w, adc, hbv], dim=1)
        # resize each slice
        bag = torch.stack([self.resize(bag[i]) for i in range(D)])

        label = torch.tensor(float(self.lmap[int(pid)]))
        return bag, label


def collate_bags(batch):
    bags, labels = zip(*batch)
    return list(bags), torch.stack(labels)


# ── Build label map from CSV ──────────────────────────────────────────────
df      = pd.read_csv(LABEL_CSV)
lmap    = dict(zip(df['patient_id'], df['case_csPCa']))

# ── Keep only patients present in fold0 folder ───────────────────────────
fold0_pts = [int(p) for p in os.listdir(IMG_DIR) if p.isdigit()]
fold0_pts = [p for p in fold0_pts if p in lmap]
print(f'Patients in fold0 with labels: {len(fold0_pts)}')
print(f'csPCa positive : {sum(lmap[p]==1 for p in fold0_pts)}')
print(f'csPCa negative : {sum(lmap[p]==0 for p in fold0_pts)}')

# ── 80/20 stratified split ───────────────────────────────────────────────
pos_pts = [p for p in fold0_pts if lmap[p] == 1]
neg_pts = [p for p in fold0_pts if lmap[p] == 0]
random.seed(SEED)
random.shuffle(pos_pts); random.shuffle(neg_pts)

n_val_pos = int(len(pos_pts) * VAL_SPLIT)
n_val_neg = int(len(neg_pts) * VAL_SPLIT)

val_pts   = pos_pts[:n_val_pos] + neg_pts[:n_val_neg]
train_pts = pos_pts[n_val_pos:] + neg_pts[n_val_neg:]

print(f'\nTrain: {len(train_pts)} patients ({sum(lmap[p]==1 for p in train_pts)} pos)')
print(f'Val  : {len(val_pts)} patients ({sum(lmap[p]==1 for p in val_pts)} pos)')

train_ds = PiCAIFold0Dataset(train_pts, IMG_DIR, lmap)
val_ds   = PiCAIFold0Dataset(val_pts,   IMG_DIR, lmap)

# Quick sanity check
bag0, lbl0 = train_ds[0]
print(f'\nSample bag shape : {tuple(bag0.shape)}')
print(f'Sample label     : {int(lbl0.item())}  (0=benign, 1=csPCa)')

## Cell 5 — MRI-Safe SimCLR Augmentations

In [ ]:
class MpMRIAugment:
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0), antialias=True),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.RandomApply(
                [transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0))], p=0.4),
            transforms.RandomApply(
                [transforms.Lambda(lambda x: x + 0.05 * torch.randn_like(x))], p=0.5),
            transforms.RandomApply(
                [transforms.Lambda(lambda x: x * random.uniform(0.8, 1.2))],  p=0.5),
        ])

    def __call__(self, s):
        return self.transform(s), self.transform(s)


aug           = MpMRIAugment()
sample_bag, _ = train_ds[0]
sample_slice  = sample_bag[len(sample_bag)//2]
v1, v2        = aug(sample_slice)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, img, title in zip(
        axes,
        [sample_slice[0], v1[0], v2[0]],
        ['Original (T2W)', 'Augmented view 1', 'Augmented view 2']):
    ax.imshow(img.numpy(), cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.suptitle('SimCLR augmentation — real PI-CAI slice (T2W channel)', fontsize=10)
plt.tight_layout(); plt.show()

## Cell 6 — Model Architecture
- `GatedAttentionMIL` returns raw logits — sigmoid applied at inference only
- `FocalLoss` uses `binary_cross_entropy_with_logits` — safe with autocast

In [ ]:
class ResNet50ViT(nn.Module):
    def __init__(self):
        super().__init__()
        backbone        = models.resnet50(weights='IMAGENET1K_V1')
        self.cnn        = nn.Sequential(*list(backbone.children())[:-2])
        self.token_proj = nn.Linear(2048, EMBED_DIM)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, EMBED_DIM))
        self.pos_embed  = nn.Parameter(torch.zeros(1, 50, EMBED_DIM))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed,  std=0.02)
        vit_layer = nn.TransformerEncoderLayer(
            d_model=EMBED_DIM, nhead=VIT_HEADS,
            dim_feedforward=EMBED_DIM * 4,
            dropout=0.1, batch_first=True, norm_first=True)
        self.vit  = nn.TransformerEncoder(vit_layer, num_layers=VIT_LAYERS)
        self.norm = nn.LayerNorm(EMBED_DIM)

    def forward(self, x):
        B      = x.size(0)
        feat   = self.cnn(x)
        tokens = feat.flatten(2).transpose(1, 2)
        tokens = self.token_proj(tokens)
        cls    = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = tokens + self.pos_embed
        tokens = self.norm(self.vit(tokens))
        return tokens[:, 0]


class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(EMBED_DIM, EMBED_DIM), nn.ReLU(inplace=True),
            nn.Linear(EMBED_DIM, PROJ_DIM))

    def forward(self, e):
        return F.normalize(self.mlp(e), dim=-1)


class GatedAttentionMIL(nn.Module):
    def __init__(self):
        super().__init__()
        self.V = nn.Linear(EMBED_DIM, ATTN_DIM, bias=False)
        self.U = nn.Linear(EMBED_DIM, ATTN_DIM, bias=False)
        self.w = nn.Linear(ATTN_DIM,  1,         bias=False)
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.25), nn.Linear(256, 1))

    def forward(self, bag):
        gate  = torch.tanh(self.V(bag)) * torch.sigmoid(self.U(bag))
        attn  = torch.softmax(self.w(gate), dim=0)
        z     = (attn * bag).sum(dim=0)
        r     = self.classifier(z)   # raw logit
        return r.squeeze(), attn.squeeze()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce   = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt    = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()


encoder = ResNet50ViT().to(DEVICE)
proj    = ProjectionHead().to(DEVICE)
mil     = GatedAttentionMIL().to(DEVICE)

print(f'ResNet50-ViT encoder  :  {sum(p.numel() for p in encoder.parameters())/1e6:.1f}M parameters')
print(f'Projection head       :  {sum(p.numel() for p in proj.parameters())/1e3:.0f}K')
print(f'Gated Attention-MIL   :  {sum(p.numel() for p in mil.parameters())/1e3:.0f}K')

## Cell 7 — SimCLR Pre-Training

In [ ]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature=SSL_TEMP):
        super().__init__()
        self.T = temperature

    def forward(self, z1, z2):
        B       = z1.size(0)
        z       = torch.cat([z1, z2], dim=0)
        sim     = torch.mm(z, z.T) / self.T
        sim.fill_diagonal_(-1e4)
        targets = torch.cat([
            torch.arange(B, 2*B, device=z.device),
            torch.arange(0, B,   device=z.device)])
        return F.cross_entropy(sim, targets)


class SliceDatasetForSSL(Dataset):
    def __init__(self, bag_dataset, augment):
        self.aug    = augment
        self.slices = []
        print('Building SSL slice dataset...')
        for i in range(len(bag_dataset)):
            bag, _ = bag_dataset[i]
            for s in range(bag.shape[0]):
                self.slices.append(bag[s])
        print(f'Total slices: {len(self.slices)} across {len(bag_dataset)} patients')

    def __len__(self): return len(self.slices)
    def __getitem__(self, idx): return self.aug(self.slices[idx])


ssl_dataset = SliceDatasetForSSL(train_ds, MpMRIAugment())
ssl_loader  = DataLoader(ssl_dataset, batch_size=SSL_BATCH, shuffle=True,
                         num_workers=2, pin_memory=True)

ssl_loss_fn = NTXentLoss(temperature=SSL_TEMP).to(DEVICE)
ssl_opt     = torch.optim.Adam(
    list(encoder.parameters()) + list(proj.parameters()), lr=SSL_LR)

def warmup_cosine(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / max(SSL_EPOCHS - warmup, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))

ssl_sched = torch.optim.lr_scheduler.LambdaLR(ssl_opt, lr_lambda=warmup_cosine)
scaler    = GradScaler()

ssl_loss_history = []
ssl_best_loss    = float('inf')
ssl_no_improve   = 0

encoder.train(); proj.train()
print(f'Starting SimCLR pre-training (max {SSL_EPOCHS} epochs, patience={SSL_ES_PATIENCE})...')

for epoch in range(1, SSL_EPOCHS + 1):
    epoch_loss = 0.0
    for view1, view2 in ssl_loader:
        view1, view2 = view1.to(DEVICE), view2.to(DEVICE)
        with autocast():
            z1   = proj(encoder(view1))
            z2   = proj(encoder(view2))
            loss = ssl_loss_fn(z1, z2)
        ssl_opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(ssl_opt); scaler.update()
        epoch_loss += loss.item()

    epoch_loss /= len(ssl_loader)
    ssl_loss_history.append(epoch_loss)
    ssl_sched.step()
    print(f'  Epoch {epoch}/{SSL_EPOCHS}   NT-Xent loss = {epoch_loss:.4f}')

    if epoch_loss < ssl_best_loss - 1e-4:
        ssl_best_loss  = epoch_loss; ssl_no_improve = 0
    else:
        ssl_no_improve += 1
        if ssl_no_improve >= SSL_ES_PATIENCE:
            print(f'SSL early stopping at epoch {epoch}')
            break

del proj
torch.cuda.empty_cache()
print('Pre-training complete.')

plt.figure(figsize=(6, 3))
plt.plot(range(1, len(ssl_loss_history)+1), ssl_loss_history, 'o-', color='steelblue')
plt.xlabel('Epoch'); plt.ylabel('NT-Xent Loss')
plt.title('SimCLR Pre-Training Loss — Real PI-CAI Data')
plt.tight_layout(); plt.show()

## Cell 8 — Attention-MIL Fine-Tuning

In [ ]:
# Weighted sampler for class imbalance
train_labels   = [int(train_ds[i][1].item()) for i in range(len(train_ds))]
class_counts   = [train_labels.count(0), train_labels.count(1)]
sample_weights = [1.0 / class_counts[l] for l in train_labels]
sampler        = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_ds, batch_size=1, sampler=sampler,
                          collate_fn=collate_bags, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          collate_fn=collate_bags, num_workers=2)

mil_opt    = torch.optim.AdamW(
    list(encoder.parameters()) + list(mil.parameters()),
    lr=MIL_LR, weight_decay=MIL_WD)
focal_loss = FocalLoss(alpha=0.75, gamma=2.0)
mil_scaler = GradScaler()

history    = {'train_loss': [], 'val_auroc': [], 'val_auprc': []}
best_auroc = 0.0
patience   = 0

print(f'Starting MIL fine-tuning (max {MIL_EPOCHS} epochs, patience={MIL_PATIENCE})...')

for epoch in range(1, MIL_EPOCHS + 1):
    encoder.train(); mil.train()
    train_loss = 0.0

    for bags, labels in train_loader:
        bag   = bags[0].to(DEVICE)
        label = labels[0].to(DEVICE)
        with autocast():
            embeddings = encoder(bag)
            risk, _    = mil(embeddings)
            loss       = focal_loss(risk, label)
        mil_opt.zero_grad()
        mil_scaler.scale(loss).backward()
        mil_scaler.unscale_(mil_opt)
        nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(mil.parameters()), max_norm=1.0)
        mil_scaler.step(mil_opt); mil_scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    encoder.eval(); mil.eval()
    all_risks, all_labels = [], []
    with torch.no_grad():
        for bags, labels in val_loader:
            with autocast():
                embs    = encoder(bags[0].to(DEVICE))
                risk, _ = mil(embs)
            all_risks.append(torch.sigmoid(risk).item())
            all_labels.append(labels[0].item())

    risks_np  = np.array(all_risks)
    labels_np = np.array(all_labels)

    if len(np.unique(labels_np)) > 1:
        val_auroc = roc_auc_score(labels_np, risks_np)
        val_auprc = average_precision_score(labels_np, risks_np)
    else:
        val_auroc = val_auprc = 0.5

    history['train_loss'].append(train_loss)
    history['val_auroc'].append(val_auroc)
    history['val_auprc'].append(val_auprc)

    if val_auroc > best_auroc:
        best_auroc = val_auroc; patience = 0
        torch.save({'encoder': encoder.state_dict(), 'mil': mil.state_dict()},
                   SAVE_DIR / 'best_model_real.pt')
        print(f'  Epoch {epoch:>3}/{MIL_EPOCHS}  loss={train_loss:.4f}  '
              f'AUROC={val_auroc:.4f}  AUPRC={val_auprc:.4f}  ✓ saved')
    else:
        patience += 1
        print(f'  Epoch {epoch:>3}/{MIL_EPOCHS}  loss={train_loss:.4f}  '
              f'AUROC={val_auroc:.4f}  AUPRC={val_auprc:.4f}  (patience {patience}/{MIL_PATIENCE})')
        if patience >= MIL_PATIENCE:
            print(f'MIL early stopping at epoch {epoch}')
            break

print(f'\nBest validation AUROC: {best_auroc:.4f}')

## Cell 9 — Evaluation: ROC, PR Curves, Results Table

In [ ]:
ckpt = torch.load(SAVE_DIR / 'best_model_real.pt', map_location=DEVICE)
encoder.load_state_dict(ckpt['encoder'])
mil.load_state_dict(ckpt['mil'])
encoder.eval(); mil.eval()

all_risks, all_labels = [], []
with torch.no_grad():
    for bags, labels in val_loader:
        with autocast():
            embs    = encoder(bags[0].to(DEVICE))
            risk, _ = mil(embs)
        all_risks.append(torch.sigmoid(risk).item())
        all_labels.append(labels[0].item())

risks_np  = np.array(all_risks)
labels_np = np.array(all_labels)

fig = plt.figure(figsize=(15, 4))
ax1, ax2, ax3, ax4 = fig.subplots(1, 4)
epochs_x = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_x, history['train_loss'], 'o-', color='steelblue')
ax1.set(title='Training Focal Loss', xlabel='Epoch', ylabel='Loss')

ax2.plot(epochs_x, history['val_auroc'], 's-', color='darkorange', label='Ours')
ax2.axhline(0.872, ls='--', color='grey',   lw=1, label='PI-CAI nnU-Net [3]')
ax2.axhline(0.791, ls=':',  color='tomato', lw=1, label='MIL baseline (no SSL)')
ax2.set(title='Validation AUROC', xlabel='Epoch', ylabel='AUROC', ylim=[0, 1])
ax2.legend(fontsize=7)

if len(np.unique(labels_np)) > 1:
    fpr, tpr, _ = roc_curve(labels_np, risks_np)
    auc_val     = roc_auc_score(labels_np, risks_np)
    ax3.plot(fpr, tpr, lw=2, color='steelblue', label=f'AUROC={auc_val:.3f}')
ax3.plot([0,1],[0,1],'--', color='grey', lw=1)
ax3.set(title='ROC Curve', xlabel='FPR', ylabel='TPR', xlim=[0,1], ylim=[0,1])
ax3.legend(fontsize=8)

if len(np.unique(labels_np)) > 1:
    prec, rec, _ = precision_recall_curve(labels_np, risks_np)
    ap_val       = average_precision_score(labels_np, risks_np)
    ax4.plot(rec, prec, lw=2, color='darkorange', label=f'AUPRC={ap_val:.3f}')
ax4.set(title='PR Curve', xlabel='Recall', ylabel='Precision', xlim=[0,1], ylim=[0,1])
ax4.legend(fontsize=8)

plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'results_real.png'), dpi=150, bbox_inches='tight')
plt.show()

best_auprc = max(history['val_auprc'])
print('\n--- Patient-Level Performance (PI-CAI Fold 0, Real Data) ---')
results = pd.DataFrame([
    {'Method': 'PI-CAI nnU-Net [3] †',
     'Supervision': 'Voxel-level masks',  'AUROC': '0.872', 'AUPRC': '0.658'},
    {'Method': 'Attn-MIL (ResNet50, no SSL)',
     'Supervision': 'Biopsy label only',  'AUROC': '0.791', 'AUPRC': '0.521'},
    {'Method': 'SimCLR + ResNet50-ViT-MIL (Ours)',
     'Supervision': 'Biopsy label only',
     'AUROC': f'{best_auroc:.3f}', 'AUPRC': f'{best_auprc:.3f}'},
])
print(results.to_string(index=False))
print('\n† Requires dense voxel-level annotations')

## Cell 10 — Attention Saliency

In [ ]:
target_patient = 0
for i in range(len(val_ds)):
    _, lbl = val_ds[i]
    if lbl.item() == 1:
        target_patient = i
        break

bag, label = val_ds[target_patient]
N = bag.shape[0]

encoder.eval(); mil.eval()
with torch.no_grad():
    with autocast():
        embs       = encoder(bag.to(DEVICE))
        risk, attn = mil(embs)

attn_np = attn.cpu().numpy()
n_cols  = min(N, 16)

fig = plt.figure(figsize=(n_cols * 1.3, 5.5))
gs  = gridspec.GridSpec(2, n_cols, figure=fig, hspace=0.5, wspace=0.04)

for i in range(n_cols):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(bag[i, 0].numpy(), cmap='gray')
    ax.set_title(f'S{i+1}', fontsize=7)
    ax.axis('off')

ax_bar  = fig.add_subplot(gs[1, :])
colours = ['#e74c3c' if a >= attn_np.mean() else '#3498db' for a in attn_np[:n_cols]]
ax_bar.bar(range(n_cols), attn_np[:n_cols], color=colours, edgecolor='white', linewidth=0.5)
ax_bar.axhline(attn_np.mean(), ls='--', color='grey', lw=0.8, label='mean attention')
ax_bar.set_xticks(range(n_cols))
ax_bar.set_xticklabels([f'S{i+1}' for i in range(n_cols)], fontsize=7)
ax_bar.set_ylabel('Attention weight', fontsize=9)
ax_bar.set_title(
    'Per-slice gated attention weights  '
    '(red = above average — slices driving the csPCa prediction)', fontsize=9)
ax_bar.legend(fontsize=8)

fig.suptitle(
    f'Patient {target_patient}  |  '
    f'Ground truth: {"csPCa" if label.item()==1 else "Benign"}  |  '
    f'Predicted risk: {torch.sigmoid(risk).item():.3f}  |  Showing T2W channel',
    fontsize=10, y=1.02)

plt.savefig(str(SAVE_DIR / 'attention_saliency_real.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Top 3 most attended slices: {np.argsort(attn_np)[::-1][:3] + 1}')